In [1]:
import cvxpy as cp
import numpy as np
import opendssdirect as dss
import yadi.dss.model as dss_model
import yadi.dss.qsts as dss_qsts
import yadi.dss.sensitivity as dss_sens
import matplotlib.pyplot as plt
import seaborn as sns

# Comparison of structure-based sensitivity recovery algorithms

In [2]:
#Load the case file
case_file = 'secondary_test_network/Master.DSS'
cktfile = r"../../test_cases/{case_file}".format(case_file=case_file)

In [52]:
qsts = dss_qsts.DSS_Timeseries(
    redirects=cktfile,
    time_step=60*60,
    simulation_steps=8739,
    )
qsts.run()

#Nodal current injections timeseries
I = qsts.currents_mvts.T

#Complex power injection timeseries
S = qsts.complex_powers_mvts.T
P = np.real(S)
Q = np.imag(S)

#Voltage phasor timeseries
V = qsts.voltages_mvts.T

#Voltage magnitude per unit (pu) timeseries
Vmags_pu = qsts.vmags_pu_mvts.T

#Line current timeseries
I_lines = qsts.line_currents_mvts.T

#Transformer current timeseries
I_xfmrs = qsts.xfmr_currents_mvts.T

print('Shapes: ',I.shape,S.shape,V.shape,Vmags_pu.shape,I_lines.shape,I_xfmrs.shape)


DSS Running file: ../../test_cases/secondary_test_network/Master.DSS
DSS Compiled Circuit: secondarytestcircuit
DSS Running file: ../../test_cases/secondary_test_network/Master.DSS
DSS Compiled Circuit: secondarytestcircuit
QSTS Initialized, Returned:  ['', '', '']


QSTS running...: 100%|██████████| 8739/8739 [36:01<00:00,  4.04it/s]

Shapes:  (118, 8739) (118, 8739) (118, 8739) (118, 8739) (100, 8739) (28, 8739)


In [42]:
sens = dss_sens.DSS_Sensitivities(cktfile,verbose=False)
svp = sens.get_svp()
svq = sens.get_svq()
svp_total,svq_total=svp,svq

DSS Running file: ../../test_cases/secondary_test_network/Master.DSS
DSS Compiled Circuit: secondarytestcircuit


In [47]:
def l2_min_norm(X,Y, lamb):
    """
    Minimum norm interpolator S (nxn) for Y = SX
    where Y (n x m) X (n x m).
    """
    n, m = Y.shape
    S = cp.Variable((n, n))
    objective = cp.Minimize(cp.norm(S, "fro"))
    constraints = [cp.norm(Y - S @ X, "fro") <= lamb]
    prob = cp.Problem(objective, constraints)
    prob.solve(verbose=True)
    return S.value

def l1_min_norm(X,Y,lamb):
    """
    Minimum l1 norm interpolator S (nxn) for Y = SX
    where Y (n x m) X (n x m).
    """
    n, m = Y.shape
    S = cp.Variable((n, n))
    objective = cp.Minimize(cp.norm(S, 1))
    constraints = [cp.norm(Y - S @ X, "fro") <= lamb]
    prob = cp.Problem(objective, constraints)
    prob.solve(verbose=True)
    return S.value

def pinv_min_norm(X,Y):
    """
    Computes the minimum norm interpolator S (n x n) for Y = SX,
    uses the tri-diagonal structural constraint. 
    """
    return Y@np.linalg.pinv(X)


def local_tri_pinv_min_norm(X,Y,lamb=2.5e-5,fill_S_matrix=True):
    """
    Generate structured tri-diagonal dynamic mode decomposition/sensitivity matrix.
    """
    d_M,d_N = X.shape
    
    #Lower diagonal (alpha), diagonal (beta) and upper diagonal (gamma) coefficients
    #Follow convention that goes lower->diag->upper
    S_matrix = np.zeros((3,d_N))

    #alpha_1 = gamma_n = 0upper and lower diagonals are zero
    S_matrix[0,0] = 0
    S_matrix[2,-1] = 0
    for i,y_i in enumerate(Y.T):
        x_i = X[:,i]
        x_ip1 = X[:,i+1]
        x_im1 = X[:,i-1]
        if(i==0): #At beginning, only contain upper tri diagonal
            S_matrix[1:,i] = y_i @ np.linalg.pinv(np.vstack((x_i,x_ip1)),lamb)
        elif(i==d_N-1): #At the end, only contain lower tri diagonal
            S_matrix[:2,i] = y_i @ np.linalg.pinv(np.vstack((x_im1,x_i)),lamb) 
        else: #contain all
            S_matrix[:,i] = y_i @ np.linalg.pinv(np.vstack((x_im1,x_i,x_ip1)),lamb)
    if fill_S_matrix:
        S_filled = np.diag(S_matrix[1,:])+np.diag(S_matrix[2,:-1],1)+np.diag(S_matrix[0,1:],-1)
        return S_filled 
    else:
        return S_matrix


def local_pinv_min_norm(X,Y,d=3,lamb=2.5e-5,fit_offset=False,fill_S_matrix=True):
    """
    Generate structured d-diagonal dynamic mode decomposition/sensitivity matrix. (Default tri-diagonal)
    """
    d_M,d_N = X.shape
    
    #Lower diagonal (alpha), diagonal (beta) and upper diagonal (gamma) coefficients
    #Follow convention that goes lower->diag->upper
    S_matrix = np.zeros((d,d_N))

    #alpha_1 = gamma_n = 0upper and lower diagonals are zero
    S_matrix[0,0] = 0
    S_matrix[2,-1] = 0
    for i,y_i in enumerate(Y.T):
        x_i = X[:,i]
        if(not fit_offset):
            if(i==0): #At beginning, only contain upper tri diagonal
                S_matrix[1:,i] = np.linalg.pinv(np.vstack((x_i.T,X[:,i+1].T)).T,lamb) @ y_i
            elif(i==d_N-1): #At the end, only contain lower tri diagonal
                S_matrix[:2,i] = np.linalg.pinv(np.vstack((X[:,i-1].T,x_i.T)).T,lamb) @ y_i
            else: #contain all
                S_matrix[:,i] = np.linalg.pinv(np.vstack((X[:,i-1].T,x_i.T,X[:,i+1].T)).T,lamb) @ y_i
        elif(fit_offset):
            if(i==0): #At beginning, only contain upper tri diagonal
                S_i = np.linalg.pinv(np.vstack([x_i.T,X[:,i+1].T,np.ones(len(x_i)).T]).T,lamb) @ y_i
                print(S_i.shape)
                S_matrix[1:,i] = S_i[:-1]
            elif(i==d_N-1): #At the end, only contain lower tri diagonal
                S_i = np.linalg.pinv(np.vstack([X[:,i-1].T,x_i.T,np.ones(len(x_i)).T]).T,lamb) @ y_i
                S_matrix[:2,i] = S_i[:-1]
            else: #contain all
                S_i = np.linalg.pinv(np.vstack([X[:,i-1].T,x_i.T,X[:,i+1].T,np.ones(len(x_i)).T]).T,lamb) @ y_i
                S_matrix[:,i] = S_i[:-1]        
    if fill_S_matrix:
        S_filled = np.diag(S_matrix[1,:])+np.diag(S_matrix[2,:-1],1)+np.diag(S_matrix[0,1:],-1)
        return S_filled 
    else:
        return S_matrix


## Generate synthetic measurements

## Algorithmic study

In [48]:
start_idx = 45
# D_diff_N = qsts.get_system_deviations(60*60)
# dP = D_diff_N['deltaP'][start_idx:,:]
# dQ = D_diff_N['deltaQ'][start_idx:,:]
# dVm = D_diff_N['deltaV'][start_idx:,:]

dP = np.diff(P)[start_idx:,:]
dQ = np.diff(Q)[start_idx:,:]
dVm = np.diff(Vmags_pu)[start_idx:,:]

#Ground truth sensitivities
svp_true = svp_total['matrix'][start_idx:,start_idx:]
svq_true = svq_total['matrix'][start_idx:,start_idx:]

print("Shapes: ",dP.shape,dQ.shape,dVm.shape,svp_true.shape,svq_true.shape)

svp_hat = pinv_min_norm(dP,dVm)
print(svp_hat.shape)

Shapes:  (73, 499) (73, 499) (73, 499) (73, 73) (73, 73)
(73, 73)


In [49]:
svp_true

array([[9.13662201e-08, 9.14251244e-08, 9.14599237e-08, ...,
        1.72529438e-09, 1.68289160e-09, 1.67880374e-09],
       [9.20587059e-08, 5.14630892e-07, 9.21178487e-08, ...,
        1.73087127e-09, 1.68879971e-09, 1.68476855e-09],
       [9.21976154e-08, 9.22186225e-08, 6.05324064e-07, ...,
        1.73200020e-09, 1.68999422e-09, 1.68597437e-09],
       ...,
       [1.30610870e-09, 1.31210588e-09, 1.31920979e-09, ...,
        6.40491496e-07, 3.20128400e-07, 3.20131476e-07],
       [1.30157984e-09, 1.30833177e-09, 1.31535890e-09, ...,
        3.18939749e-07, 4.86617705e-07, 4.38242408e-07],
       [1.30109631e-09, 1.30792794e-09, 1.31494673e-09, ...,
        3.18801583e-07, 4.38037521e-07, 4.64532472e-07]])

In [50]:
svp_hat

array([[ 4.53301173e-17, -6.46919706e-05, -5.14759560e-05, ...,
        -2.51034058e-05,  9.75672059e-03, -5.51154463e-03],
       [ 6.38580906e-17,  1.72321574e-05, -1.60599474e-04, ...,
        -2.11954806e-05,  1.01539410e-02, -6.15998408e-03],
       [ 8.67509934e-17, -7.39047854e-05,  6.05269713e-05, ...,
        -7.55205060e-06,  1.16896042e-02, -1.23539968e-02],
       ...,
       [ 4.96360584e-17, -2.40432227e-05, -2.18410514e-04, ...,
         3.02409955e-04,  1.02416029e-02, -9.76522318e-03],
       [ 5.11454033e-17, -3.25940681e-05, -1.85353262e-04, ...,
         1.66490668e-04,  7.13051858e-03, -1.20274592e-02],
       [ 6.05395069e-17, -3.10982126e-05, -1.81527656e-04, ...,
         1.65446449e-04,  8.28335948e-03, -1.31097871e-02]])

In [70]:
np.linalg.norm(svp_hat-svp_true)/np.linalg.norm(svp_true)

0.00028217878700218034

In [100]:
np.hstack((dP,dQ)).shape

(719, 146)

## Generate synthetic data

In [69]:
np.random.seed(2023)
n_nodes = svp_true.shape[0]
m_measurements = 8760
dP = np.random.normal(loc=0,scale=np.abs(np.mean(P)),size=(n_nodes,m_measurements))
dV = svp_true@dP
dV_obs = dV + np.random.normal(loc=0,scale=np.abs(np.mean(dV)),size=(n_nodes,m_measurements))
svp_hat = dV_obs@np.linalg.pinv(dP)

#print relative error:
print("Relative error of SVP estimate: {}".format(np.linalg.norm(svp_hat-svp_true)/np.linalg.norm(svp_true)))

Relative error of SVP estimate: 0.00028217878700218034
